In [ ]:
import pandas as pd

path = "../data/"

In [ ]:
df = pd.read_csv(
    path + "sentiment140.csv",
    encoding="latin-1",
    header=None
)

In [ ]:
df.columns = ["target", "id", "date", "flag", "user", "text"]
df.head()

,target,id,date,flag,user,text
0,4,2046362316,Fri Jun 05 12:04:14 PDT 2009,NO_QUERY,winecliQ,@VERGEwine Thx for the #ff lovin - now if that...
1,0,1984664187,Sun May 31 15:33:25 PDT 2009,NO_QUERY,XeBeans,another WONDERFUL weekend! i almost hate to go...
2,0,2187830458,Mon Jun 15 20:52:20 PDT 2009,NO_QUERY,ACTinglikeamama,belly pushing against jeans - only three month...
3,0,1992717155,Mon Jun 01 09:12:29 PDT 2009,NO_QUERY,scarletsighting,missing London. with all honesty.
4,4,1792666808,Wed May 13 23:50:50 PDT 2009,NO_QUERY,sambirchall,Wow!!! I can go on twitwer on my iPod that's ...


In [ ]:
df = df[['target', 'text']]

In [ ]:
df['label'] = df['target'].replace({
    0: 'negative',
    4: 'positive'
})

In [ ]:
df.head()
df['label'].value_counts()

label
negative    10000
positive    10000
Name: count, dtype: int64

In [ ]:
df = df.groupby('label', group_keys=False).sample(
    n=10000,
    random_state=42
)

In [ ]:
df['label'].value_counts()

label
negative    10000
positive    10000
Name: count, dtype: int64

In [ ]:
df_air = pd.read_csv("../data/tweets2.csv")
df_air.head()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [ ]:
df_air = df_air[['text', 'airline_sentiment']]

In [ ]:
df_air.head()

,text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [ ]:
df_air = df_air.rename(columns={
    'airline_sentiment': 'label'
})

In [ ]:
df_air.head()

,text,label
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [ ]:
df_air['label'].value_counts()

label
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64

In [ ]:
df_neu = df_air[df_air['label'] == 'neutral']

df_neu_upsampled = df_neu.sample(
    n=8000, 
    replace=True,   
    random_state=42
)

In [ ]:
df_pos = df[df['label'] == 'positive'].sample(n=8000, random_state=42)
df_neg = df[df['label'] == 'negative'].sample(n=8000, random_state=42)

df_final = pd.concat([df_pos, df_neg, df_neu_upsampled])

In [ ]:
df_final['label'].value_counts()

label
positive    8000
negative    8000
neutral     8000
Name: count, dtype: int64

In [ ]:
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)   
    text = re.sub(r"@\w+", "", text)      
    text = re.sub(r"#", "", text)      
    text = re.sub(r"[^a-z\s]", " ", text) 
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
df_final['clean_text'] = df_final['text'].astype(str).apply(preprocess)

In [ ]:
df_final[['text','clean_text']].head()

,text,clean_text
393952,Just cleaned out my baby's beta fish tank. Now...,just cleaned out my baby s beta fish tank now ...
1545423,"morning everyone - v.tired, need to get the mo...",morning everyone v tired need to get the morni...
446201,going back to santos in a couple of hours,going back to santos in a couple of hours
458206,@NickWayne87 lmaoo exactly !! and thank you th...,lmaoo exactly and thank you thats much appreci...
828350,Vvv that was rachel not me rachel's vagina sme...,vvv that was rachel not me rachel s vagina sme...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    stop_words='english'
)

X = vectorizer.fit_transform(df_final['clean_text'])
y = df_final['label']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.72      0.68      0.70      1600
     neutral       0.89      0.92      0.91      1600
    positive       0.68      0.70      0.69      1600

    accuracy                           0.77      4800
   macro avg       0.76      0.77      0.77      4800
weighted avg       0.76      0.77      0.77      4800



In [51]:
from sklearn.naive_bayes import MultinomialNB
nb = MultinomialNB()
nb.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [52]:
from sklearn.svm import LinearSVC
svm = LinearSVC()
svm.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

In [53]:
from sklearn.metrics import classification_report

for name, m in {
    "LogReg": model,
    "NB": nb,
    "SVM": svm
}.items():
    preds = m.predict(X_test)
    print(f"\n{name}\n", classification_report(y_test, preds))


LogReg
               precision    recall  f1-score   support

    negative       0.72      0.68      0.70      1600
     neutral       0.89      0.92      0.91      1600
    positive       0.68      0.70      0.69      1600

    accuracy                           0.77      4800
   macro avg       0.76      0.77      0.77      4800
weighted avg       0.76      0.77      0.77      4800


NB
               precision    recall  f1-score   support

    negative       0.67      0.71      0.69      1600
     neutral       0.92      0.87      0.90      1600
    positive       0.67      0.66      0.66      1600

    accuracy                           0.75      4800
   macro avg       0.75      0.75      0.75      4800
weighted avg       0.75      0.75      0.75      4800


SVM
               precision    recall  f1-score   support

    negative       0.71      0.68      0.69      1600
     neutral       0.91      0.93      0.92      1600
    positive       0.67      0.69      0.68      1600



In [54]:
import pickle

pickle.dump(model, open("../model.pkl", "wb"))
pickle.dump(vectorizer, open("../vectorizer.pkl", "wb"))

In [63]:
import numpy as np

labels = model.classes_   

def predict_with_threshold(texts, threshold=0.5):
    texts_clean = [preprocess(t) for t in texts]
    X = vectorizer.transform(texts_clean)
    
    probs = model.predict_proba(X)
    
    final_preds = []
    
    for p in probs:
        max_prob = np.max(p)
        label = labels[np.argmax(p)]
        
        if max_prob < threshold:
            final_preds.append("neutral")
        else:
            final_preds.append(label)
    
    return final_preds

In [64]:
sample = [
    "I love this product",
    "This is terrible",
    "It is okay, nothing special"
]

print(predict_with_threshold(sample))

['positive', 'negative', 'positive']


In [66]:
def credibility(followers, likes):
    if followers > 500 and likes > 10:
        return "Authentic"
    return "Suspicious"

In [67]:
from collections import Counter

counts = Counter(preds)

total = len(preds)

for k in counts:
    print(k, round(counts[k] / total * 100, 2), "%")

neutral 33.98 %
positive 34.29 %
negative 31.73 %


In [93]:
def get_rss_data():
    feed = feedparser.parse(
        "https://www.reddit.com/r/technology/.rss",
        request_headers={'User-Agent': 'sentiment-app'}
    )
    return [entry.title for entry in feed.entries]

In [69]:
pip install feedparser

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6090 sha256=be1502579b6335ad4b49442197da47e0518951b835d7f27a42b1a6208f0eee59
  Stored in directory: /Users/harshitasaxena/Library/Caches/pip/wheels/e3/43/83/0f6e317d0698ac38ee6a5b6e214019c167057916a11bad91ab
Successfully built sgmllib3k
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [feedparser]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [71]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [79]:
import os
from dotenv import load_dotenv

load_dotenv()

NEWS_API_KEY = os.getenv("NEWS_API_KEY")